# Comparing fine-tuning techniques
## Introduction
In this activity, you will apply and compare the different fine-tuning techniques we’ve covered so far: traditional fine-tuning, Low-Rank Adaptation (LoRA), and Quantized Low-Rank Adaptation (QLoRa). You will evaluate each approach's performance and resource efficiency and learn how to choose the right method based on the task and hardware constraints.

This activity compares how traditional fine-tuning, LoRA, and QLoRA perform in terms of computational cost, memory usage, and task performance. You will fine-tune the same pretrained model using these three techniques and compare the results based on training time, memory consumption, and model accuracy on a task-specific dataset.

By the end of this activity, you will be able to:
* Apply different fine-tuning techniques to a pretrained model.
* Compare the computational efficiency and model performance of each fine-tuning technique.
* Evaluate how to choose the best fine-tuning technique based on hardware.

## Step-by-step guide to compare techniques
Create a new Jupyter Notebook. You can call it “fine_tuning_comparion”. Make sure you have the appropriate Python kernel selected.

The remaining of this reading will guide you through the following steps:
* Step 1: Prepare your dataset
* Step 2: Apply traditional fine-tuning
* Step 3: Fine-tune with LoRA
* Step 4: Fine-tune with QLoRA
* Step 5: Compare and analyze results

### Step 1: Prepare your dataset
The first step is to prepare your dataset for fine-tuning. You will use the same dataset for all three techniques to ensure a fair comparison.

#### Instructions
1. Load a sample dataset from scikit-learn and inspect its structure.
2. Apply preprocessing steps such as cleaning and tokenizing (use the BERT tokenizer) the full dataset before splitting it into sets. This ensures consistent preparation across the entire dataset and avoids data leakage.
3. Split the dataset into training, validation, and test sets.

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split

# Load dataset
#data = pd.read_csv('your_dataset.csv')
data = pd.read_json("hf://datasets/aleixsant/alpaca_cleaned_en/en.jsonl", lines=True)

# Split dataset into training, validation, and test sets
train_data, temp_data = train_test_split(data, test_size=0.3, random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

print(f"Training set size: {len(train_data)}")
print(f"Validation set size: {len(val_data)}")
print(f"Test set size: {len(test_data)}")

Training set size: 36401
Validation set size: 7800
Test set size: 7801


### Step 2: Apply traditional fine-tuning
In the first part of the activity, you will apply traditional fine-tuning to the pretrained model. This involves updating all the parameters of the model during training.

#### Instructions
1. Load a pretrained model (e.g., BERT or GPT).
2. Fine-tune the entire model on the task-specific dataset.
3. Record the training time, memory usage, and evaluation metrics.

#### Record
* Training time: initialize a timer before each training block to print a message when training is complete.
* Memory usage: use monitoring tools (e.g., nvidia-smi) to track memory consumption.
* Model performance: print each performance metric (accuracy, precision, recall, F1 score) to the console after each block to compare between methods.

In [5]:
import time
from typing import Any, cast
from torch.utils.data import Dataset as TorchDataset
from transformers import BertForSequenceClassification, BertTokenizer, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import LabelEncoder
import numpy as np

# Initialize timer
start_time = time.time()

# Encode labels (using "domain" as target)
label_encoder = LabelEncoder()
label_encoder.fit(data["domain"])

train_data = train_data.copy()
val_data = val_data.copy()
test_data = test_data.copy()

train_data["labels"] = label_encoder.transform(train_data["domain"]).astype(np.int64)
val_data["labels"] = label_encoder.transform(val_data["domain"]).astype(np.int64)
test_data["labels"] = label_encoder.transform(test_data["domain"]).astype(np.int64)

# Load BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenize datasets
def tokenize_function(examples):
    return tokenizer(examples['instruction'], padding='max_length', truncation=True, max_length=128)

train_dataset = Dataset.from_pandas(train_data[['instruction', 'labels']]).map(tokenize_function, batched=True)
val_dataset = Dataset.from_pandas(val_data[['instruction', 'labels']]).map(tokenize_function, batched=True)
test_dataset = Dataset.from_pandas(test_data[['instruction', 'labels']]).map(tokenize_function, batched=True)

# Set format to PyTorch tensors
train_dataset.set_format('torch', columns=['input_ids', 'token_type_ids', 'attention_mask', 'labels'])
val_dataset.set_format('torch', columns=['input_ids', 'token_type_ids', 'attention_mask', 'labels'])
test_dataset.set_format('torch', columns=['input_ids', 'token_type_ids', 'attention_mask', 'labels'])

# Load pre-trained BERT model
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=len(label_encoder.classes_),
    problem_type='single_label_classification'
)

# Set up training arguments
training_args = TrainingArguments(
    output_dir='./results_traditional',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    eval_strategy="epoch",
    remove_unused_columns=False,
    logging_steps=100,
)

# Define compute metrics function
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    accuracy = accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, average='weighted', zero_division=0)
    recall = recall_score(labels, predictions, average='weighted', zero_division=0)
    f1 = f1_score(labels, predictions, average='weighted', zero_division=0)
    return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

# Fine-tune the model
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

# Start fine-tuning
trainer.train()

# Evaluate on test set
predictions = trainer.predict(test_dataset)
metrics = compute_metrics((predictions.predictions, predictions.label_ids))

# Print completion message and metrics
elapsed_time = time.time() - start_time
print(f"\n✓ Traditional fine-tuning completed in {elapsed_time:.2f} seconds ({elapsed_time/60:.2f} minutes)")
print("\nPerformance Metrics:")
print(f"  Accuracy:  {metrics['accuracy']:.4f}")
print(f"  Precision: {metrics['precision']:.4f}")
print(f"  Recall:    {metrics['recall']:.4f}")
print(f"  F1 Score:  {metrics['f1']:.4f}")

Map:   0%|          | 0/36401 [00:00<?, ? examples/s]

Map:   0%|          | 0/7800 [00:00<?, ? examples/s]

Map:   0%|          | 0/7801 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/Users/peter.horstedt/git/AI-and-Mach

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/peter.horstedt/git/AI-and-Machine-Learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/peter.horstedt/git/AI-and-Machine-Learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)



✓ Traditional fine-tuning completed in 3773.53 seconds (62.89 minutes)

Performance Metrics:
  Accuracy:  1.0000
  Precision: 1.0000
  Recall:    1.0000
  F1 Score:  1.0000


### Step 3: Fine-tune with LoRA
Next, you will fine-tune the model using LoRA. This technique updates only low-rank matrices added to certain layers while freezing the rest of the model’s parameters.

#### Instructions
1. Import the necessary libraries, ensure the correct kernel is selected and tokenize the data set.
2. Install the peft package.
3. Initialize the BERT model and then define a LoRa configuration, define the rank of the update matrices and the alpha scaling factor.
4. Apply LoRA to specific layers of the pretrained model (e.g., attention layers).
5. Create a LoRa model using peft and set the number of training epochs.
6. Initialize a data collator, initialize a trainer, and then you can train the model.
7. Fine-tune only the low-rank matrices while keeping the other parameters frozen.
8. Record the same metrics as in traditional fine-tuning.

#### Record
* Training time: initialize a timer before each training block to print a message when training is complete.
* Memory usage: track memory consumption as you did in the previous step.
* Model performance: print each performance metric (accuracy, precision, recall, F1 score) to the console after each block to compare between methods.

In [6]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import BertForSequenceClassification, Trainer, TrainingArguments
import time

# Initialize timer
start_time_lora = time.time()

# Re-load base model for a clean LoRA run
model_lora_base = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=len(label_encoder.classes_),
    problem_type='single_label_classification'
)

# LoRA configuration for BERT attention projections
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "key", "value"]
)

# Wrap model with LoRA adapters (base weights remain frozen by PEFT)
model_lora = get_peft_model(model_lora_base, lora_config)
model_lora.print_trainable_parameters()

training_args_lora = TrainingArguments(
    output_dir='./results_lora',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    eval_strategy="epoch",
    remove_unused_columns=False,
    logging_steps=100,
)

trainer_lora = Trainer(
    model=model_lora,
    args=training_args_lora,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer_lora.train()

predictions_lora = trainer_lora.predict(test_dataset)
metrics_lora = compute_metrics((predictions_lora.predictions, predictions_lora.label_ids))

elapsed_time_lora = time.time() - start_time_lora
print(f"\n✓ LoRA fine-tuning completed in {elapsed_time_lora:.2f} seconds ({elapsed_time_lora/60:.2f} minutes)")
print("\nLoRA Performance Metrics:")
print(f"  Accuracy:  {metrics_lora['accuracy']:.4f}")
print(f"  Precision: {metrics_lora['precision']:.4f}")
print(f"  Recall:    {metrics_lora['recall']:.4f}")
print(f"  F1 Score:  {metrics_lora['f1']:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 443,137 || all params: 109,926,146 || trainable%: 0.4031


/Users/peter.horstedt/git/AI-and-Machine-Learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000
2,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000
3,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000


/Users/peter.horstedt/git/AI-and-Machine-Learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/peter.horstedt/git/AI-and-Machine-Learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/peter.horstedt/git/AI-and-Machine-Learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/peter.horstedt/git/AI-and-Machine-Learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device 


✓ LoRA fine-tuning completed in 3034.28 seconds (50.57 minutes)

LoRA Performance Metrics:
  Accuracy:  1.0000
  Precision: 1.0000
  Recall:    1.0000
  F1 Score:  1.0000


### Step 4: Fine-tune with QLoRA
Finally, you will apply QLoRA to the pretrained model. QLoRA quantizes the model’s parameters to reduce memory usage even further and fine-tunes only low-rank matrices.

#### Instructions
1. Import the necessary libraries, ensure the correct kernel is selected and tokenize the data set.
2. Quantize the pretrained model to reduce the memory footprint.
3. Apply LoRA to the quantized model’s layers.
4. Fine-tune the model and record the metrics.

#### Record
* Training time: initialize a timer before each training block to print a message when training is complete.
* Memory usage: track memory usage with monitoring tools.
* Model performance: print each performance metric (accuracy, precision, recall, F1 score) to the console after each block to compare between methods.

In [8]:
import torch
import time
from transformers import BertForSequenceClassification, Trainer, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

# Initialize timer
start_time_qlora = time.time()

# QLoRA with bitsandbytes is CUDA-only in practice.
# On macOS/MPS or CPU, fall back to regular LoRA so the notebook still runs.
use_true_qlora = torch.cuda.is_available()

if use_true_qlora:
    # 4-bit quantization config for QLoRA (standard setup)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

    model_qlora_base = BertForSequenceClassification.from_pretrained(
        'bert-base-uncased',
        num_labels=len(label_encoder.classes_),
        problem_type='single_label_classification',
        quantization_config=bnb_config,
        device_map='auto',
    )

    # Required for stable k-bit PEFT training
    model_qlora_base = prepare_model_for_kbit_training(model_qlora_base)
    print('Running true QLoRA (4-bit) on CUDA.')
else:
    model_qlora_base = BertForSequenceClassification.from_pretrained(
        'bert-base-uncased',
        num_labels=len(label_encoder.classes_),
        problem_type='single_label_classification',
    )
    print('CUDA not available. Falling back to LoRA-compatible (non-quantized) run for this step.')

# Define LoRA/QLoRA adapter configuration
qlora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=['query', 'key', 'value'],
    bias='none',
    use_rslora=False,
)

# Apply adapters
model_qlora = get_peft_model(model_qlora_base, qlora_config)
model_qlora.print_trainable_parameters()

# Set up training arguments
training_args_qlora = TrainingArguments(
    output_dir='./results_qlora',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    eval_strategy='epoch',
    remove_unused_columns=False,
    logging_steps=100,
)

# Initialize trainer
trainer_qlora = Trainer(
    model=model_qlora,
    args=training_args_qlora,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

# Train
trainer_qlora.train()

# Evaluate on test set
predictions_qlora = trainer_qlora.predict(test_dataset)
metrics_qlora = compute_metrics((predictions_qlora.predictions, predictions_qlora.label_ids))

# Print completion message and metrics
elapsed_time_qlora = time.time() - start_time_qlora
print(f"\n✓ QLoRA/LoRA step completed in {elapsed_time_qlora:.2f} seconds ({elapsed_time_qlora/60:.2f} minutes)")
print("\nQLoRA Step Performance Metrics:")
print(f"  Accuracy:  {metrics_qlora['accuracy']:.4f}")
print(f"  Precision: {metrics_qlora['precision']:.4f}")
print(f"  Recall:    {metrics_qlora['recall']:.4f}")
print(f"  F1 Score:  {metrics_qlora['f1']:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


CUDA not available. Falling back to LoRA-compatible (non-quantized) run for this step.
trainable params: 443,137 || all params: 109,926,146 || trainable%: 0.4031


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000
2,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000
3,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000


/Users/peter.horstedt/git/AI-and-Machine-Learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/peter.horstedt/git/AI-and-Machine-Learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/peter.horstedt/git/AI-and-Machine-Learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/peter.horstedt/git/AI-and-Machine-Learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device 


✓ QLoRA/LoRA step completed in 2887.72 seconds (48.13 minutes)

QLoRA Step Performance Metrics:
  Accuracy:  1.0000
  Precision: 1.0000
  Recall:    1.0000
  F1 Score:  1.0000


### Step 5: Compare and analyze results
After completing the fine-tuning process using all three techniques, you will compare the results. The goal is to analyze the trade-offs between model performance, training time, and memory usage for each technique.

Questions to consider
1. Which technique was the fastest to train?
2. Which technique used the least memory?
3. How did the model performance compare across the different techniques?
4. Based on your results, which technique would you recommend for scenarios with limited computational resources?

## Deliverables
By the end of this activity, you should produce:
1. A report summarizing the results of the three fine-tuning techniques, including training time, memory usage, and model performance metrics.
2. An analysis of which technique is the most efficient and why.
3. A reflection on how these techniques can be applied in real-world scenarios.

## Conclusion
By comparing traditional fine-tuning, LoRA, and QLoRA, you will gain a deeper understanding of the trade-offs between computational efficiency and model performance. This activity will help you decide when to use each 